# 15-lira-wn, LiRA shadows, weight-noise comparator, audit extensions

Settings: **Accelerator GPU T4 x2**, **Internet On**, secret `kaggle_dpfl-braintumor` attached, input `brain-mri-curated-v1` added, and the outputs of 07-week3-core, 08-fed-seeds-a, 09-fed-seeds-b-unit and 10-unit-udir added through Add Input, Notebook Output.

Runs code tag `v0.6-lira`. Trains the weight-noise comparator (5 selection runs, then seeds and shadows by a fixed rule) and 64 LiRA shadow models, then saves logits of every audited model, the skull-border analysis and probe baselines. About 5.5 hours on two GPUs.

Generated by `scripts/make_launcher.py --kind lira`, edit that script rather than this notebook.

In [ ]:
import os, sys, glob, json, shutil, subprocess, datetime
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
CODE_REF = "v0.6-lira"                 # tag of the code to run
GRIDS = ["week03_central", "week04_unit_central", "week04_unit_central_ext", "week03_fed_seeds_a", "week03_fed_seeds_b", "week04_unit_fed"]
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
os.environ["GIT_PUSH_URL"] = f"https://{token}@github.com/{USER}/{REPO}.git"
CLEAN = f"https://github.com/{USER}/{REPO}.git"
CODE, RES = "/tmp/code", "/tmp/results"
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")

def run(cmd, cwd=None, check=True, env=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

for d in (CODE, RES):
    shutil.rmtree(d, ignore_errors=True)
    run(["git", "clone", "-q", os.environ["GIT_PUSH_URL"], d])
    run(["git", "remote", "set-url", "origin", CLEAN], cwd=d)
run(["git", "checkout", "-q", CODE_REF], cwd=CODE)
run(["git", "config", "user.name", USER], cwd=RES)
run(["git", "config", "user.email", EMAIL], cwd=RES)
COMMIT = run(["git", "rev-parse", "HEAD"], cwd=CODE).strip()
print(run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"])[-500:])

hits = glob.glob("/kaggle/input/**/brain_mri_curated_v1.npz", recursive=True)
if not hits:
    raise SystemExit("Dataset not attached. In the right panel use Add Input, Your Datasets, "
                     "brain-mri-curated-v1, wait for it to mount, then rerun this cell.")
DATA = os.path.dirname(hits[0])
import torch
print("code", CODE_REF, COMMIT[:10], "| data", DATA)
print("torch", torch.__version__, "| GPUs", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
import shlex
OLD_AUDIT = ["week03_central", "week04_unit_central", "week04_unit_central_ext", "week03_fed_seeds_a", "week03_fed_seeds_b", "week04_unit_fed"]

# Checkpoints come from earlier launchers' outputs attached as inputs (Add Input, Notebook Output).
XAI_GRIDS = ["week03_central", "week04_unit_central", "week04_unit_central_ext", "week03_fed_seeds_a"]
pts = glob.glob("/kaggle/input/**/checkpoints/*.pt", recursive=True)
by_dir = {}
for p in pts:
    by_dir.setdefault(p.split("/checkpoints/")[0], []).append(p)
for d, v in sorted(by_dir.items()):
    print(f"{len(v):4d} checkpoints in {d}")
if not pts:
    raise SystemExit("No checkpoints found. Add Input, Notebook Output, and attach the outputs of "
                     "07-week3-core, 08-fed-seeds-a, 09-fed-seeds-b-unit and 10-unit-udir, then rerun this cell.")

In [ ]:
# Test suite on the Kaggle image, report pushed to results/tests/.
sys.path.insert(0, CODE)
from scripts.run_grid import push_run
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider"],
                   cwd=CODE, capture_output=True, text=True)
report = f"date {TODAY}\ncode {CODE_REF} {COMMIT}\ntorch {torch.__version__}\nreturncode {r.returncode}\n\n" + r.stdout[-6000:]
print(report[-2500:])
os.makedirs(f"{RES}/results/tests", exist_ok=True)
open(f"{RES}/results/tests/{TODAY}_{CODE_REF}_pytest.txt", "w").write(report)
print("push", push_run(RES, "results/tests", f"Test report {CODE_REF}"))
assert r.returncode == 0, "Tests failed, stop here and report"

In [ ]:
# Weight-noise sigma selection at seed 0 (5 runs), then the seeds and shadow grids by the fixed rule (D45).
os.makedirs("/kaggle/working/logs", exist_ok=True)
def shards(grids, tag):
    procs = []
    for shard in (0, 1):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
        script = " && ".join(
            f"{sys.executable} scripts/run_grid.py --grid configs/grids/{g}.yaml --data-dir {DATA} "
            f"--results-repo {RES} --shard {shard} --nshards 2 --push --ckpt-dir /kaggle/working/checkpoints"
            for g in grids)
        log = open(f"/kaggle/working/logs/{tag}_shard{shard}.log", "w")
        procs.append(subprocess.Popen(["bash", "-c", script], cwd=CODE, env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    print(tag, "exit codes", codes)
    for shard in (0, 1):
        print(f"----- {tag} shard {shard} tail")
        print(open(f"/kaggle/working/logs/{tag}_shard{shard}.log").read().replace(token, "***")[-1500:])

shards(["week07_wn_select"], "select")
print(run([sys.executable, "scripts/make_shadow_grid.py", "wn", "--results", f"{RES}/results", "--out", RES], cwd=CODE))
for g in ("week07_wn_seeds", "week07_wn_shadows"):
    shutil.copy(f"{RES}/configs/grids/{g}.yaml", f"{CODE}/configs/grids/{g}.yaml")
print("push", push_run(RES, "configs/grids", "Weight-noise grids from week07_wn_select"))

In [ ]:
# Training: weight-noise seeds (5 per sigma), LiRA shadows (64), weight-noise shadows (16 per sigma).
# About 4.8 hours on two GPUs (DP shadows about 8 min each). Finished runs are skipped if this cell is rerun.
shards(["week07_wn_seeds", "week07_lira_shadows", "week07_wn_shadows"], "train")

In [ ]:
# Audit: logits for every target and shadow, border analysis for the explanation set,
# loss attack and probe for the new targets, probe baselines once. No training.
AUDIT_TARGETS = OLD_AUDIT + ["week07_wn_select", "week07_wn_seeds"]
procs = []
for shard in (0, 1):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard))
    common = ["--data-dir", DATA, "--ckpt-root", "/kaggle/input", "/kaggle/working/checkpoints",
              "--results-repo", RES, "--shard", str(shard), "--nshards", "2", "--push"]
    a = [sys.executable, "scripts/run_audit.py", "--grids", *AUDIT_TARGETS, "--xai-grids", *XAI_GRIDS,
         "--parts", "mia", "probe", "logits", "border", "--probe-baselines", *common]
    b = [sys.executable, "scripts/run_audit.py", "--grids", "week07_lira_shadows", "week07_wn_shadows",
         "--parts", "logits", *common]
    script = " ".join(map(shlex.quote, a)) + " && " + " ".join(map(shlex.quote, b))
    log = open(f"/kaggle/working/logs/audit_shard{shard}.log", "w")
    procs.append(subprocess.Popen(["bash", "-c", script], cwd=CODE, env=env, stdout=log, stderr=subprocess.STDOUT))
codes = [p.wait() for p in procs]
print("exit codes", codes)
for shard in (0, 1):
    txt = open(f"/kaggle/working/logs/audit_shard{shard}.log").read().replace(token, "***")
    print(f"----- audit shard {shard}")
    print("\n".join(l for l in txt.splitlines() if "finished" in l or "missing" in l or "FAILED" in l or "baseline" in l))

In [ ]:
# Tables: weight-noise utility, stratified loss attack, LiRA, border and consistency. Pushed to results/tables/.
print(run([sys.executable, "scripts/summarise.py", "--runs", f"{RES}/results/runs",
           "--grids", "configs/grids/week07_wn_seeds.yaml", "--out", f"{RES}/results/tables/week07_wn_seeds_summary.csv",
           "--md", f"{RES}/results/tables/week07_wn_seeds_summary.md"], cwd=CODE))
print(run([sys.executable, "scripts/mia_tables.py", "--results", f"{RES}/results",
           "--out", f"{RES}/results/tables/week07"], cwd=CODE)[-6000:])
print(run([sys.executable, "scripts/xai_tables.py", "--results", f"{RES}/results",
           "--out", f"{RES}/results/tables/week07_xai_border"], cwd=CODE)[-4000:])
for t in ("pretrained", "random"):
    p = f"{RES}/results/audit/_baselines/probe_{t}.json"
    print("probe baseline", t, open(p).read() if os.path.exists(p) else "missing")
print("push", push_run(RES, "results/tables", f"Tables {CODE_REF}"))